In [1]:
import pandas as pd
import re
import math
import json
import requests
from bs4 import BeautifulSoup
from io import StringIO
from variables import TOUR_NAME, POKEDATA_CSV, DAY1_ROUNDS, CATEGORY

In [2]:
def clean_name(input_string):
    result = re.sub(r'\s*\[.*?\]\s*', '', input_string)
    result = re.sub(r'STATIC SEATING \(\d+\)\s*', '', result)
    result = re.sub(r'>.*?>', '', result)
    result = re.sub(r'>TABLE \d+ ', '', result)
    return result

In [3]:
pairings_df = pd.read_csv(StringIO(requests.get(POKEDATA_CSV).content.decode('utf-8')), sep='\t', header=None, encoding='utf-8')
pairings_df.rename(columns={0:'Player',1:'Opponent',2:'Result',3:'Points',4:'Round'}, inplace=True)
pairings_df['Player'] = pairings_df['Player'].apply(clean_name)
pairings_df['Opponent'] = pairings_df['Opponent'].apply(clean_name)
pairings_df = pairings_df[(pairings_df['Opponent'] != 'BYE') & (pairings_df['Opponent'] != 'LATE')]

In [4]:
pairings_df

,Player,Opponent,Result,Points,Round
0,Isaiah Bradner,Xuanwen Zhang,W,3,1
1,Isaiah Bradner,niccolò genna,W,6,2
2,Isaiah Bradner,Rune Heiremans,L,6,3
3,Isaiah Bradner,Koen Smid,W,9,4
4,Isaiah Bradner,Timo Dell,W,12,5
...,...,...,...,...,...
6504,Adam Marshall,Toni Haverinen,T,7,4
6505,Adam Marshall,Arttu Norrlin,W,10,5
6506,Adam Marshall,Marcin Nowacki,W,13,6
6507,Adam Marshall,Ajay Sridhar,W,16,7


In [5]:
deck_df = pd.read_excel(f'standings/{TOUR_NAME}_standings.xlsx', sheet_name=CATEGORY)
deck_df['Placement'] = deck_df['Placement'].apply(lambda x: "Top {}".format(pow(2, math.ceil(math.log(x, 2)))))
deck_df['Day 2'] = deck_df['Player'].map(lambda player: pairings_df.groupby('Player')['Round'].count().get(player, 0) > DAY1_ROUNDS)


In [6]:
# Check missing players
player_index = 0
for player in pairings_df['Player'].unique():
    if player not in deck_df['Player'].unique():
        print(player_index, player)
    player_index+=1

814 Adam Marshall


In [8]:
deck_dict = deck_df.set_index('Player')['Deck'].to_dict()

In [10]:
with pd.ExcelWriter(f'datasets/{TOUR_NAME}_{CATEGORY}.xlsx') as writer:
    deck_df.to_excel(writer, sheet_name='decks', index=False)
    pairings_df.to_excel(writer, sheet_name='pairings', index=False)